# Explorando o lakehouse

Tour pelas camadas do lakehouse, com **uma verificação por célula** (✅/❌, esperado × encontrado) e um quadro-resumo no fim.
Por enquanto **bronze** (fase 1) e **silver** (fase 2). A gold entra na próxima fase.

**Regra de leitura (F3):** toda tabela Delta é lida pelo `_delta_log` (`DeltaTable` → DuckDB, função `ler_delta`), nunca pelos `.parquet` da pasta.

In [1]:
import os
from pathlib import Path

# O notebook roda de dentro de notebooks/; o código do projeto está na raiz.
if Path.cwd().name == "notebooks":
    os.chdir("..")

import pandas as pd
from deltalake import DeltaTable
from src.comum import carregar_config, conectar_duckdb, caminho_tabela, ler_delta, registrar_delta

cfg = carregar_config()
con = conectar_duckdb(cfg)
con.execute("SET enable_progress_bar = false")  # a barra trava o notebook no VS Code
TABELAS = [t["tabela"] for t in cfg["bronze"]["tabelas"]]
RASTREIO = ["_arquivo_origem", "_data_carga", "_carregado_em"]

resultados = []
camada_atual = "Bronze"

def verificar(nome, esperado, encontrado, ok=None):
    """Registra uma verificação e mostra ✅/❌ com o esperado e o encontrado."""
    ok = (esperado == encontrado) if ok is None else ok
    resultados.append({"camada": camada_atual, "verificação": nome, "esperado": esperado, "encontrado": encontrado, "ok": "✅" if ok else "❌"})
    print(f"{'✅' if ok else '❌'} {nome}\n   esperado:   {esperado}\n   encontrado: {encontrado}")

def ultima_carga(tabela):
    dt = DeltaTable(str(caminho_tabela(cfg, "bronze", tabela)))
    return max(p["_data_carga"] for p in dt.partitions())

def bronze(tabela, carga=None):
    return ler_delta(con, caminho_tabela(cfg, "bronze", tabela), carga=carga)

def silver(tabela):
    return ler_delta(con, caminho_tabela(cfg, "silver", tabela))

def um(sql):
    """Primeira linha de uma consulta."""
    return con.sql(sql).fetchall()[0]

def fmt_n(x):
    return f"{x:,}".replace(",", ".")

def fmt_us(x):
    return "US$ " + f"{x:,.2f}".replace(",", "X").replace(".", ",").replace("X", ".")

print(f"lake: {cfg['paths']['lake']}")

lake: C:\Dashboard_Lakehouse\lake


## Bronze

Os 7 CSVs da Dunnhumby como chegaram: **tudo texto**, com `_arquivo_origem`, `_data_carga` (partição, data local de São Paulo) e `_carregado_em`.
Cada carga é uma partição; rodar de novo no mesmo dia substitui só aquela partição.

### Controle de cargas
Uma linha por tabela por execução. Abaixo, a execução mais recente de cada tabela.

In [2]:
controle = bronze("_controle_cargas")
ultimas = con.sql("""
    SELECT * FROM controle
    QUALIFY row_number() OVER (PARTITION BY tabela ORDER BY carregado_em DESC) = 1
    ORDER BY tabela
""").df()
ultimas

,tabela,arquivo,data_carga,carregado_em,linhas_lidas,linhas_gravadas,colunas_esperadas_ok,status,mensagem
0,campaign_desc,campaign_desc.csv,2026-10-28,2026-10-28 16:32:25.028676-03:00,30,30,True,sucesso,None
1,campaign_table,campaign_table.csv,2026-10-28,2026-10-28 16:32:24.871739-03:00,7208,7208,True,sucesso,None
2,coupon,coupon.csv,2026-10-28,2026-10-28 16:32:25.173663-03:00,124548,124548,True,sucesso,None
3,coupon_redempt,coupon_redempt.csv,2026-10-28,2026-10-28 16:32:25.952967-03:00,2318,2318,True,sucesso,None
4,hh_demographic,hh_demographic.csv,2026-10-28,2026-10-28 16:32:24.642972-03:00,801,801,True,sucesso,None
5,product,product.csv,2026-10-28,2026-10-28 16:32:23.999695-03:00,92353,92353,True,sucesso,None
6,transaction_data,transaction_data.csv,2026-10-28,2026-10-28 16:32:03.219490-03:00,2595732,2595732,True,sucesso,None


### 1. Última carga das 7 tabelas com `sucesso`

In [3]:
sucesso = sorted(ultimas.loc[ultimas["status"] == "sucesso", "tabela"])
verificar("última carga com sucesso", f"{len(TABELAS)} de {len(TABELAS)}", f"{len(sucesso)} de {len(TABELAS)}")

✅ última carga com sucesso
   esperado:   7 de 7
   encontrado: 7 de 7


### 2. Contagem de cada tabela = `linhas_gravadas` no controle
Contagem feita **pela tabela Delta**, na partição da última carga, contra o que a ingestão registrou.

In [4]:
linhas = []
for t in TABELAS:
    carga = ultima_carga(t)
    contagem = bronze(t, carga).aggregate("count(*)").fetchall()[0][0]
    registro = ultimas.loc[ultimas["tabela"] == t].iloc[0]
    linhas.append({"tabela": t, "carga": carga, "contagem_delta": contagem,
                   "linhas_gravadas_controle": int(registro["linhas_gravadas"]),
                   "carga_do_controle": registro["data_carga"]})
contagens = pd.DataFrame(linhas)
display(contagens)
iguais = ((contagens["contagem_delta"] == contagens["linhas_gravadas_controle"])
          & (contagens["carga"] == contagens["carga_do_controle"])).sum()
verificar("contagem Delta = linhas_gravadas do controle", f"{len(TABELAS)} de {len(TABELAS)} tabelas", f"{iguais} de {len(TABELAS)} tabelas")

,tabela,carga,contagem_delta,linhas_gravadas_controle,carga_do_controle
0,transaction_data,2026-10-28,2595732,2595732,2026-10-28
1,product,2026-10-28,92353,92353,2026-10-28
2,hh_demographic,2026-10-28,801,801,2026-10-28
3,campaign_table,2026-10-28,7208,7208,2026-10-28
4,campaign_desc,2026-10-28,30,30,2026-10-28
5,coupon,2026-10-28,124548,124548,2026-10-28
6,coupon_redempt,2026-10-28,2318,2318,2026-10-28


✅ contagem Delta = linhas_gravadas do controle
   esperado:   7 de 7 tabelas
   encontrado: 7 de 7 tabelas


### 3. Colunas presentes = colunas esperadas do `config.yaml` (+ as 3 de rastreio)

In [5]:
divergentes = []
for item in cfg["bronze"]["tabelas"]:
    presentes = [c for c in bronze(item["tabela"]).columns]
    if presentes != item["colunas"] + RASTREIO:
        divergentes.append(item["tabela"])
verificar("colunas = config + rastreio", "nenhuma tabela divergente", ", ".join(divergentes) or "nenhuma tabela divergente")

✅ colunas = config + rastreio
   esperado:   nenhuma tabela divergente
   encontrado: nenhuma tabela divergente


### 4. Colunas de dado 100% texto

In [6]:
tipos = []
for t in TABELAS:
    for coluna, tipo in zip(bronze(t).columns, bronze(t).types):
        if coluna not in RASTREIO:
            tipos.append({"tabela": t, "coluna": coluna, "tipo": str(tipo)})
tipos = pd.DataFrame(tipos)
display(tipos.groupby("tipo").size().rename("colunas").to_frame())
verificar("colunas de dado como texto", f"{len(tipos)} de {len(tipos)} (100%)",
          f"{(tipos['tipo'] == 'VARCHAR').sum()} de {len(tipos)} ({(tipos['tipo'] == 'VARCHAR').mean():.0%})")

,colunas
tipo,
VARCHAR,41


✅ colunas de dado como texto
   esperado:   41 de 41 (100%)
   encontrado: 41 de 41 (100%)


### 5. Rastreio sem nulos (`_arquivo_origem`, `_data_carga`, `_carregado_em`), na tabela inteira

In [7]:
nulos = []
for t in TABELAS:
    soma = " + ".join(f"count(*) FILTER (WHERE {c} IS NULL)" for c in RASTREIO)
    nulos.append({"tabela": t, "nulos_rastreio": bronze(t).aggregate(soma).fetchall()[0][0]})
nulos = pd.DataFrame(nulos)
display(nulos)
verificar("nulos nas colunas de rastreio", 0, int(nulos["nulos_rastreio"].sum()))

,tabela,nulos_rastreio
0,transaction_data,0
1,product,0
2,hh_demographic,0
3,campaign_table,0
4,campaign_desc,0
5,coupon,0
6,coupon_redempt,0


✅ nulos nas colunas de rastreio
   esperado:   0
   encontrado: 0


### 6. Gabarito
Números do projeto de RAG (relatório de qualidade): `transaction_data` com **2.595.732** linhas brutas e `product` com **92.353** produtos.

In [8]:
n = contagens.set_index("tabela")["contagem_delta"]
verificar("transaction_data = gabarito", f"{2_595_732:,}".replace(",", "."), f"{n['transaction_data']:,}".replace(",", "."))

✅ transaction_data = gabarito
   esperado:   2.595.732
   encontrado: 2.595.732


In [9]:
verificar("product = gabarito", f"{92_353:,}".replace(",", "."), f"{n['product']:,}".replace(",", "."))

✅ product = gabarito
   esperado:   92.353
   encontrado: 92.353


### 7. Versões Delta de cada tabela
Cada gravação gera uma versão nova; as antigas continuam disponíveis para *time travel* (`espiar.py ... --versao N`) até o `vacuum`.

In [10]:
versoes = []
for t in TABELAS + ["_controle_cargas"]:
    hist = DeltaTable(str(caminho_tabela(cfg, "bronze", t))).history()
    ultima = max(hist, key=lambda h: h["version"])
    versoes.append({"tabela": t, "versao_atual": ultima["version"], "versoes": len(hist),
                    "ultima_operacao": ultima["operation"],
                    "predicado": (ultima.get("operationParameters") or {}).get("predicate")})
versoes = pd.DataFrame(versoes)
versoes

,tabela,versao_atual,versoes,ultima_operacao,predicado
0,transaction_data,1,2,WRITE,_data_carga = '2026-10-28'
1,product,1,2,WRITE,_data_carga = '2026-10-28'
2,hh_demographic,1,2,WRITE,_data_carga = '2026-10-28'
3,campaign_table,1,2,WRITE,_data_carga = '2026-10-28'
4,campaign_desc,1,2,WRITE,_data_carga = '2026-10-28'
5,coupon,1,2,WRITE,_data_carga = '2026-10-28'
6,coupon_redempt,1,2,WRITE,_data_carga = '2026-10-28'
7,_controle_cargas,13,14,WRITE,NaN


**Por que a regra F3 existe:** depois de duas cargas no mesmo dia, a pasta de `transaction_data` guarda os arquivos das duas versões.
Ler a pasta inteira com `*.parquet` contaria em dobro; a tabela Delta conta só os arquivos da versão atual.

In [11]:
pasta = caminho_tabela(cfg, "bronze", "transaction_data")
carga = ultima_carga("transaction_data")
arquivos_na_pasta = len(list((pasta / f"_data_carga={carga}").glob("*.parquet")))
arquivos_da_versao = len(DeltaTable(str(pasta)).file_uris(file_pruning_predicate=[("_data_carga", "=", carga)]))
pela_pasta = con.sql(f"SELECT count(*) FROM read_parquet('{(pasta / f'_data_carga={carga}').as_posix()}/*.parquet')").fetchall()[0][0]
print(f"partição {carga}: {arquivos_na_pasta} arquivos .parquet na pasta, {arquivos_da_versao} na versão atual (pelo _delta_log)")
print("lendo a pasta direto (ERRADO, só para mostrar):", f"{pela_pasta:,}".replace(",", "."), "linhas")
verificar("leitura Delta não conta arquivos antigos",
          f"{2_595_732:,}".replace(",", "."),
          f"{bronze('transaction_data', carga).aggregate('count(*)').fetchall()[0][0]:,}".replace(",", "."))

partição 2026-10-28: 2 arquivos .parquet na pasta, 1 na versão atual (pelo _delta_log)
lendo a pasta direto (ERRADO, só para mostrar): 5.191.464 linhas


✅ leitura Delta não conta arquivos antigos
   esperado:   2.595.732
   encontrado: 2.595.732


## Silver

Construída por `python -m src.silver.build` a partir da **carga mais recente** da bronze, reconstruída por inteiro a cada execução (P8).
- `silver/clean/`: uma tabela por arquivo, tipada e com nomes em snake_case; o que falha numa regra vai para `silver/_rejected/` com o `_motivo`.
- `silver/context/`: o star schema (`fact_sales` + `dim_product`, `dim_store`, `dim_household`, `dim_date`), com membro desconhecido `-1` em cada dimensão.

As regras estão declaradas no `config.yaml` (seção `silver`) e explicadas na S9 do documento de decisões.

In [12]:
camada_atual = "Silver"
carga_tx = ultima_carga("transaction_data")
registrar_delta(con, "bronze_tx", caminho_tabela(cfg, "bronze", "transaction_data"), carga=carga_tx)
for t in ["clean/transaction_data", "clean/product", "clean/hh_demographic",
          "_rejected/transaction_data", "_rejected/product", "_rejected/hh_demographic"]:
    registrar_delta(con, t.replace("/", "_").lstrip("_"), caminho_tabela(cfg, "silver", t))
for t in ["sales/fact_sales", "sales/dim_store", "customer/dim_household", "product/dim_product", "shared/dim_date"]:
    registrar_delta(con, t.split("/")[1], caminho_tabela(cfg, "silver", f"context/{t}"))

# Controle da silver: a execução mais recente de cada tabela.
controle_silver = silver("_controle_silver")
con.sql("""
    SELECT tabela, data_carga_bronze, processado_em, linhas_entrada, linhas_saida, linhas_rejeitadas,
           round(100 * pct_rejeitado, 2) AS pct, status, mensagem
    FROM controle_silver
    QUALIFY row_number() OVER (PARTITION BY tabela ORDER BY processado_em DESC) = 1
    ORDER BY camada, tabela
""").df()

,tabela,data_carga_bronze,processado_em,linhas_entrada,linhas_saida,linhas_rejeitadas,pct,status,mensagem
0,clean/hh_demographic,2026-10-28,2026-10-28 19:51:20.427387-03:00,801,801,0,0.00,sucesso,nenhuma rejeitada
1,clean/product,2026-10-28,2026-10-28 19:51:20.427387-03:00,92353,92353,0,0.00,sucesso,nenhuma rejeitada
2,clean/transaction_data,2026-10-28,2026-10-28 19:51:20.427387-03:00,2595732,2581266,14466,0.56,sucesso,"quantity_zero=14,466"
3,context/customer/dim_household,2026-10-28,2026-10-28 19:51:20.427387-03:00,<NA>,2501,0,0.00,sucesso,NaN
4,context/product/dim_product,2026-10-28,2026-10-28 19:51:20.427387-03:00,<NA>,92354,0,0.00,sucesso,NaN
5,context/sales/dim_store,2026-10-28,2026-10-28 19:51:20.427387-03:00,<NA>,583,0,0.00,sucesso,NaN
6,context/sales/fact_sales,2026-10-28,2026-10-28 19:51:20.427387-03:00,2581266,2581266,0,0.00,sucesso,"orfaos_produto=0, orfaos_loja=0, orfaos_cliente=0"
7,context/shared/dim_date,2026-10-28,2026-10-28 19:51:20.427387-03:00,<NA>,712,0,0.00,sucesso,NaN


### 8. `transaction_data`: bronze = clean + rejeitadas

In [13]:
n_bronze = um("SELECT count(*) FROM bronze_tx")[0]
n_clean = um("SELECT count(*) FROM clean_transaction_data")[0]
n_rej = um("SELECT count(*) FROM rejected_transaction_data")[0]
verificar("transaction_data: bronze = clean + rejeitadas",
          "2.595.732 = 2.581.266 + 14.466",
          f"{fmt_n(n_bronze)} = {fmt_n(n_clean)} + {fmt_n(n_rej)}",
          ok=(n_bronze, n_clean, n_rej) == (2_595_732, 2_581_266, 14_466) and n_bronze == n_clean + n_rej)

✅ transaction_data: bronze = clean + rejeitadas
   esperado:   2.595.732 = 2.581.266 + 14.466
   encontrado: 2.595.732 = 2.581.266 + 14.466


### 9. Rejeitadas por motivo

In [14]:
motivos = con.sql("SELECT _motivo, count(*) AS linhas FROM rejected_transaction_data GROUP BY 1 ORDER BY 2 DESC").fetchall()
display(pd.DataFrame(motivos, columns=["_motivo", "linhas"]))
verificar("rejeitadas por motivo", "quantity_zero: 14.466 (0,56%)",
          "; ".join(f"{m}: {fmt_n(n)} (" + f"{100 * n / n_bronze:.2f}".replace(".", ",") + "%)" for m, n in motivos))

,_motivo,linhas
0,quantity_zero,14466


✅ rejeitadas por motivo
   esperado:   quantity_zero: 14.466 (0,56%)
   encontrado: quantity_zero: 14.466 (0,56%)


### 10. `product` e `hh_demographic`: tudo aceito

In [15]:
rejeitadas = um("SELECT count(*) FROM rejected_product")[0] + um("SELECT count(*) FROM rejected_hh_demographic")[0]
verificar("product e hh_demographic", "92.353 e 801, 0 rejeitadas",
          f"{fmt_n(um('SELECT count(*) FROM clean_product')[0])} e {fmt_n(um('SELECT count(*) FROM clean_hh_demographic')[0])}, {rejeitadas} rejeitadas")

✅ product e hh_demographic
   esperado:   92.353 e 801, 0 rejeitadas
   encontrado: 92.353 e 801, 0 rejeitadas


### 11. `fact_sales`: linhas e faturamento
O total é o da bronze (US$ 8.057.463,08) menos os US$ 19,28 das linhas com quantidade zero, que foram rejeitadas.

In [16]:
linhas_fato, soma_fato = um("SELECT count(*), sum(sales_value) FROM fact_sales")
verificar("fact_sales: linhas / soma de sales_value", "2.581.266 / US$ 8.057.443,80", f"{fmt_n(linhas_fato)} / {fmt_us(soma_fato)}")

✅ fact_sales: linhas / soma de sales_value
   esperado:   2.581.266 / US$ 8.057.443,80
   encontrado: 2.581.266 / US$ 8.057.443,80


### 12. Mercadoria × não mercadoria
`is_merchandise` vem da `dim_product`: departamentos KIOSK-GAS, MISC SALES TRAN, MISC. TRANS., COUP/STR & MFG ou commodity COUPON/MISC ITEMS = não mercadoria.
Mercadoria tem que bater com os números "limpos" do projeto de RAG.

In [17]:
merc = um("SELECT count(*), sum(sales_value) FROM fact_sales WHERE is_merchandise")
verificar("mercadoria (is_merchandise = true)", "2.548.662 linhas, US$ 7.376.322,37", f"{fmt_n(merc[0])} linhas, {fmt_us(merc[1])}")

✅ mercadoria (is_merchandise = true)
   esperado:   2.548.662 linhas, US$ 7.376.322,37
   encontrado: 2.548.662 linhas, US$ 7.376.322,37


In [18]:
nao_merc = um("SELECT count(*), sum(sales_value) FROM fact_sales WHERE NOT is_merchandise")
verificar("não mercadoria (is_merchandise = false)", "32.604 linhas, US$ 681.121,43", f"{fmt_n(nao_merc[0])} linhas, {fmt_us(nao_merc[1])}")

✅ não mercadoria (is_merchandise = false)
   esperado:   32.604 linhas, US$ 681.121,43
   encontrado: 32.604 linhas, US$ 681.121,43


### 13. Marcações: item grátis e estorno de desconto (em mercadoria / no total)

In [19]:
gratis = um("SELECT count(*) FILTER (WHERE is_free_item AND is_merchandise), count(*) FILTER (WHERE is_free_item) FROM fact_sales")
verificar("is_free_item em mercadoria / total", "3.994 / 4.451", f"{fmt_n(gratis[0])} / {fmt_n(gratis[1])}")

✅ is_free_item em mercadoria / total
   esperado:   3.994 / 4.451
   encontrado: 3.994 / 4.451


In [20]:
estorno = um("SELECT count(*) FILTER (WHERE is_refund_discount AND is_merchandise), count(*) FILTER (WHERE is_refund_discount) FROM fact_sales")
verificar("is_refund_discount em mercadoria / total", "2 / 2", f"{fmt_n(estorno[0])} / {fmt_n(estorno[1])}")

✅ is_refund_discount em mercadoria / total
   esperado:   2 / 2
   encontrado: 2 / 2


**Por que 2 / 2, e não os 2 / 10 do perfil (achado da fase 2, corrigido na S9.4 em 29/10/2026):** o perfil da bronze contou as 10 linhas com `retail_disc > 0` **antes** da regra da quantidade zero.
As 10 são todas de mercadoria, e **8 delas têm `quantity = 0`**, então são rejeitadas (`quantity_zero`, S9.1) e não chegam à `fact_sales`. Sobram 2, como no gabarito do RAG.
A conciliação abaixo confere, por outro caminho, que nenhum estorno sumiu.

In [21]:
bronze_estorno = um("SELECT count(*) FROM bronze_tx WHERE CAST(RETAIL_DISC AS DECIMAL(18,4)) > 0")[0]
rej_estorno = um("SELECT count(*) FROM rejected_transaction_data WHERE CAST(RETAIL_DISC AS DECIMAL(18,4)) > 0 AND _motivo = 'quantity_zero'")[0]
verificar("estornos: fact_sales + rejeitadas (quantity_zero) = bronze", "2 + 8 = 10",
          f"{estorno[1]} + {rej_estorno} = {bronze_estorno}",
          ok=(estorno[1], rej_estorno, bronze_estorno) == (2, 8, 10))

✅ estornos: fact_sales + rejeitadas (quantity_zero) = bronze
   esperado:   2 + 8 = 10
   encontrado: 2 + 8 = 10


### 14. Soma semanal de `sales_value`: `fact_sales` × bronze (conferência por outro caminho, S6)
O lado da bronze é calculado aqui, com SQL próprio direto do texto da bronze, **sem reaproveitar o código da silver**: converte o texto e deixa de fora a quantidade zero (a única regra que rejeitou linhas).

In [22]:
semanal = con.sql("""
    WITH b AS (
        SELECT CAST(WEEK_NO AS INTEGER) AS week_no, sum(CAST(SALES_VALUE AS DECIMAL(18,4))) AS bronze
        FROM bronze_tx
        WHERE CAST(QUANTITY AS INTEGER) <> 0
        GROUP BY 1
    ), s AS (
        SELECT week_no, sum(sales_value) AS silver FROM fact_sales GROUP BY 1
    )
    SELECT coalesce(b.week_no, s.week_no) AS week_no, b.bronze, s.silver,
           abs(coalesce(b.bronze, 0) - coalesce(s.silver, 0)) AS diferenca
    FROM b FULL JOIN s ON b.week_no = s.week_no
    ORDER BY week_no
""").df()
display(semanal.sort_values("diferenca", ascending=False).head(3))
maior = semanal["diferenca"].max()
verificar("soma semanal fact_sales × bronze", "102 semanas, diferença ≤ US$ 0,01",
          f"{len(semanal)} semanas, diferença máxima {fmt_us(maior)}",
          ok=len(semanal) == 102 and maior <= 0.01)

,week_no,bronze,silver,diferenca
0,1,5211.16,5211.16,0.0
64,65,87396.48,87396.48,0.0
74,75,83162.42,83162.42,0.0


✅ soma semanal fact_sales × bronze
   esperado:   102 semanas, diferença ≤ US$ 0,01
   encontrado: 102 semanas, diferença máxima US$ 0,00


### 15. Vendas apontando para o membro desconhecido (-1)

In [23]:
orfaos = um("SELECT count(*) FILTER (WHERE product_id = -1), count(*) FILTER (WHERE store_id = -1), count(*) FILTER (WHERE household_key = -1) FROM fact_sales")
verificar("vendas com produto / loja / cliente = -1", "0 / 0 / 0", " / ".join(fmt_n(x) for x in orfaos))

✅ vendas com produto / loja / cliente = -1
   esperado:   0 / 0 / 0
   encontrado: 0 / 0 / 0


### 16. Chaves das dimensões: sem nulo e sem repetição

In [24]:
chaves = {"dim_product": "product_id", "dim_store": "store_id", "dim_household": "household_key", "dim_date": "day"}
problemas = pd.DataFrame([
    {"dimensão": d, "chave": k,
     "nulas": um(f"SELECT count(*) FROM {d} WHERE {k} IS NULL")[0],
     "repetidas": um(f"SELECT count(*) - count(DISTINCT {k}) FROM {d}")[0]}
    for d, k in chaves.items()])
display(problemas)
verificar("chaves das dim_: nulas e repetidas", "0 e 0", f"{problemas['nulas'].sum()} e {problemas['repetidas'].sum()}")

,dimensão,chave,nulas,repetidas
0,dim_product,product_id,0,0
1,dim_store,store_id,0,0
2,dim_household,household_key,0,0
3,dim_date,day,0,0


✅ chaves das dim_: nulas e repetidas
   esperado:   0 e 0
   encontrado: 0 e 0


### 17. Tamanho das dimensões (sem o membro -1)

In [25]:
tamanhos = [um(f"SELECT count(*) FROM {d} WHERE {k} <> -1")[0] for d, k in list(chaves.items())[:3]]
verificar("dim_product / dim_store / dim_household", "92.353 / 582 / 2.500", " / ".join(fmt_n(x) for x in tamanhos))

✅ dim_product / dim_store / dim_household
   esperado:   92.353 / 582 / 2.500
   encontrado: 92.353 / 582 / 2.500


In [26]:
verificar("dim_household com has_demographics = true", "801",
          fmt_n(um("SELECT count(*) FROM dim_household WHERE has_demographics")[0]))

✅ dim_household com has_demographics = true
   esperado:   801
   encontrado: 801


### 18. `dim_date`
`week_no` vem do arquivo (as semanas começam no `day` 6, 13, 20…), `date = 2016-01-01 + (day − 1)` e `day_of_cycle` vai de 1 a 7 a partir do primeiro dia da semana.

In [27]:
dias, sem_min, sem_max, dist_sem = um("SELECT count(*), min(week_no), max(week_no), count(DISTINCT week_no) FROM dim_date WHERE day <> -1")
ciclo6, data1 = um("SELECT max(day_of_cycle) FILTER (WHERE day = 6), max(date) FILTER (WHERE day = 1) FROM dim_date")
verificar("dim_date", "711 dias, semanas 1 a 102 (102 distintas), day 6 com day_of_cycle = 1, day 1 = 2016-01-01",
          f"{dias} dias, semanas {sem_min} a {sem_max} ({dist_sem} distintas), day 6 com day_of_cycle = {ciclo6}, day 1 = {data1}")

✅ dim_date
   esperado:   711 dias, semanas 1 a 102 (102 distintas), day 6 com day_of_cycle = 1, day 1 = 2016-01-01
   encontrado: 711 dias, semanas 1 a 102 (102 distintas), day 6 com day_of_cycle = 1, day 1 = 2016-01-01


## Quadro-resumo

In [28]:
resumo = pd.DataFrame(resultados)[["ok", "camada", "verificação", "esperado", "encontrado"]]
for camada, grupo in resumo.groupby("camada", sort=False):
    print(f"{camada}: {(grupo['ok'] == '✅').sum()} de {len(grupo)} ✅")
print(f"Total: {(resumo['ok'] == '✅').sum()} de {len(resumo)} verificações ✅")
resumo

Bronze: 8 de 8 ✅


Silver: 15 de 15 ✅
Total: 23 de 23 verificações ✅


,ok,camada,verificação,esperado,encontrado
0,✅,Bronze,última carga com sucesso,7 de 7,7 de 7
1,✅,Bronze,contagem Delta = linhas_gravadas do controle,7 de 7 tabelas,7 de 7 tabelas
2,✅,Bronze,colunas = config + rastreio,nenhuma tabela divergente,nenhuma tabela divergente
3,✅,Bronze,colunas de dado como texto,41 de 41 (100%),41 de 41 (100%)
4,✅,Bronze,nulos nas colunas de rastreio,0,0
5,✅,Bronze,transaction_data = gabarito,2.595.732,2.595.732
6,✅,Bronze,product = gabarito,92.353,92.353
7,✅,Bronze,leitura Delta não conta arquivos antigos,2.595.732,2.595.732
8,✅,Silver,transaction_data: bronze = clean + rejeitadas,2.595.732 = 2.581.266 + 14.466,2.595.732 = 2.581.266 + 14.466
9,✅,Silver,rejeitadas por motivo,"quantity_zero: 14.466 (0,56%)","quantity_zero: 14.466 (0,56%)"
